# Scalar Chebyshev2 vs Trapezoid: EuRoC Aggressive Windows

This notebook mirrors `scalar_quadrature_random_degree4_polynomials.ipynb`, but each scalar function is derived from the most aggressive 1-second `gyro_norm` window in one merged EuRoC dataset. The 1-second window is fit with Chebyshev2 (`N=50`), cached as node values, and the quadrature experiment samples only the middle 200 ms interval `[0.4, 0.6]`. Set `USE_LAMBDA1_TIKHONOV = True` to add the Chebyshev2 derivative-matrix Tikhonov penalty to the noisy Chebyshev fits.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "python" / "imuFactors").exists():
    REPO_ROOT = Path("/Users/dellaert/git/imuFactors")
PYTHON_DIR = REPO_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

import imuFactors.euroc as euroc
import imuFactors.scalar_quadrature as scalar_quadrature
import imuFactors.spectral as spectral
import imuFactors.spectrogram as spectrogram

plt.rcParams.update({"figure.dpi": 120})


In [ ]:
DATA_DIR = REPO_ROOT / "data" / "euroc"
DATA_FILES = euroc.discover_euroc_files(DATA_DIR)
if not DATA_FILES:
    raise FileNotFoundError(f"No EuRoC CSV files found in {DATA_DIR}")

CACHE_PATH = DATA_DIR / "euroc_aggressive_gyro_norm_cheb2_n50_windows.npz"
FORCE_REBUILD_CACHE = False

SIGNAL_COLUMN = "gyro_norm"
REFERENCE_BASIS = "chebyshev2"
REFERENCE_NODE_COUNT = 50
REFERENCE_WINDOW_SECONDS = 1.0
REFERENCE_LAMBDA1 = 0.0
FULL_INTERVAL = (0.0, 1.0)
MIDDLE_INTERVAL = (0.4, 0.6)

SAMPLE_COUNTS = [10, 20, 30, 40, 50]
CHEBYSHEV_NODES = np.arange(2, 11)
NOISE_FRACTIONS = np.array([
    0.0, 0.025, 0.05, 0.06, 0.075, 0.10,
    0.12, 0.15, 0.17, 0.20, 0.225,
])
NUM_SEEDS = 100
RANDOM_SEED = 20260523
EVALUATION_COUNT = 151

LAMBDA1_SWEEP_SAMPLE_COUNT = 40
LAMBDA1_SWEEP_GRID = np.r_[0.0, np.logspace(-8, 1, 19)]

# Switch for Chebyshev2 Tikhonov regularization with derivative matrix D.
USE_LAMBDA1_TIKHONOV = True
LAMBDA1_TIKHONOV = 3.1622776601683794e-3
EXPERIMENT_LAMBDA1 = LAMBDA1_TIKHONOV if USE_LAMBDA1_TIKHONOV else 0.0

print(f"Found {len(DATA_FILES)} EuRoC CSV files in {DATA_DIR}")
print(f"cache: {CACHE_PATH}")
print(f"Chebyshev2 fit lambda1: {EXPERIMENT_LAMBDA1:g}")


In [ ]:
def _dataset_name(path: Path) -> str:
    return path.stem.removeprefix("euroc_")


def _selected_window_row(path: Path, result: spectrogram.SpectrogramFit) -> dict:
    scores = spectrogram.aggressive_window_scores(result)
    window_index = int(np.nanargmax(scores))
    nrms = spectrogram.normalized_rmse(result)
    return {
        "dataset": _dataset_name(path),
        "csv_path": str(path),
        "window_index": window_index,
        "start_seconds": spectrogram.window_start_seconds(result, window_index),
        "aggressive_score": float(scores[window_index]),
        "activity": float(result.activity[window_index]),
        "high_order_ratio": float(result.high_order_ratio[window_index]),
        "normalized_rmse": float(nrms[window_index]),
        "raw_samples": result.samples[window_index, :, 0].astype(float),
        "node_values": result.coeffs[window_index, :, 0].astype(float),
        "sample_seconds": np.linspace(0.0, result.window_seconds, result.sample_count),
    }


def build_aggressive_window_cache(data_files: list[Path], cache_path: Path) -> dict:
    rows = []
    for path in data_files:
        print(f"fitting {_dataset_name(path)}...")
        result = spectrogram.fit_spectral_windows(
            path,
            coefficient_count=REFERENCE_NODE_COUNT,
            basis=REFERENCE_BASIS,
            columns=[SIGNAL_COLUMN],
            window_seconds=REFERENCE_WINDOW_SECONDS,
            lambda1=REFERENCE_LAMBDA1,
        )
        rows.append(_selected_window_row(path, result))

    sample_seconds = rows[0]["sample_seconds"]
    if any(row["sample_seconds"].shape != sample_seconds.shape for row in rows):
        raise ValueError("Selected windows do not all have the same sample count")

    payload = {
        "datasets": np.array([row["dataset"] for row in rows]),
        "csv_paths": np.array([row["csv_path"] for row in rows]),
        "window_indices": np.array([row["window_index"] for row in rows], dtype=int),
        "start_seconds": np.array([row["start_seconds"] for row in rows], dtype=float),
        "aggressive_scores": np.array([row["aggressive_score"] for row in rows], dtype=float),
        "activity": np.array([row["activity"] for row in rows], dtype=float),
        "high_order_ratio": np.array([row["high_order_ratio"] for row in rows], dtype=float),
        "normalized_rmse": np.array([row["normalized_rmse"] for row in rows], dtype=float),
        "raw_samples": np.stack([row["raw_samples"] for row in rows]),
        "node_values": np.stack([row["node_values"] for row in rows]),
        "sample_seconds": sample_seconds,
        "node_seconds": spectral.chebyshev2_points(REFERENCE_NODE_COUNT, FULL_INTERVAL),
        "reference_node_count": np.array(REFERENCE_NODE_COUNT, dtype=int),
        "reference_window_seconds": np.array(REFERENCE_WINDOW_SECONDS, dtype=float),
        "middle_interval": np.array(MIDDLE_INTERVAL, dtype=float),
    }
    np.savez_compressed(cache_path, **payload)
    return payload


def load_aggressive_window_cache(cache_path: Path) -> dict:
    with np.load(cache_path, allow_pickle=False) as archive:
        return {key: archive[key] for key in archive.files}


In [ ]:
if FORCE_REBUILD_CACHE or not CACHE_PATH.exists():
    cache = build_aggressive_window_cache(DATA_FILES, CACHE_PATH)
else:
    cache = load_aggressive_window_cache(CACHE_PATH)

metadata = pd.DataFrame(
    {
        "dataset": cache["datasets"],
        "window_index": cache["window_indices"],
        "start_s": cache["start_seconds"],
        "aggressive_score": cache["aggressive_scores"],
        "activity": cache["activity"],
        "high_order_ratio": cache["high_order_ratio"],
        "normalized_rmse": cache["normalized_rmse"],
    }
).sort_values("aggressive_score", ascending=False)
metadata


In [ ]:
FUNCTIONS = [
    scalar_quadrature.scalar_function_from_chebyshev2_nodes(
        f"{dataset} aggressive gyro_norm", node_values, FULL_INTERVAL
    )
    for dataset, node_values in zip(cache["datasets"], cache["node_values"])
]

pd.DataFrame(
    cache["node_values"],
    columns=[f"cgl{k}" for k in range(int(cache["reference_node_count"]))],
    index=[function.name for function in FUNCTIONS],
).iloc[:, :10]


In [ ]:
def plot_cached_window_previews(cache: dict) -> go.Figure:
    datasets = list(cache["datasets"])
    rows = len(datasets)
    fig = make_subplots(
        rows=rows,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.012,
        subplot_titles=datasets,
    )
    dense_seconds = np.linspace(0.0, 1.0, 401)
    for row, (dataset, node_values, raw_samples) in enumerate(
        zip(cache["datasets"], cache["node_values"], cache["raw_samples"]), start=1
    ):
        function = scalar_quadrature.scalar_function_from_chebyshev2_nodes(
            str(dataset), node_values, FULL_INTERVAL
        )
        fig.add_trace(
            go.Scatter(
                x=cache["sample_seconds"],
                y=raw_samples,
                mode="markers",
                marker=dict(size=3),
                name=f"{dataset} samples",
                showlegend=False,
            ),
            row=row,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=dense_seconds,
                y=function.value(dense_seconds),
                mode="lines",
                line=dict(width=2),
                name=f"{dataset} Chebyshev2 fit",
                showlegend=False,
            ),
            row=row,
            col=1,
        )
        fig.add_vrect(
            x0=MIDDLE_INTERVAL[0],
            x1=MIDDLE_INTERVAL[1],
            fillcolor="rgba(80, 140, 255, 0.16)",
            line_width=0,
            row=row,
            col=1,
        )
        fig.update_yaxes(title_text="gyro", row=row, col=1)
    fig.update_xaxes(title_text="seconds inside selected 1 s window", row=rows, col=1)
    fig.update_layout(
        title="Selected aggressive 1-second windows; shaded region is the sampled 200 ms interval",
        height=max(700, 145 * rows),
        margin=dict(l=70, r=30, t=80, b=55),
    )
    return fig

plot_cached_window_previews(cache).show()


In [ ]:
node_ranges = {
    sample_count: CHEBYSHEV_NODES
    for sample_count in SAMPLE_COUNTS
}
pd.DataFrame(
    {
        "N": list(node_ranges),
        "m_min": [values[0] for values in node_ranges.values()],
        "m_max": [values[-1] for values in node_ranges.values()],
        "sqrt_N": [np.sqrt(sample_count) for sample_count in node_ranges],
        "num_m": [len(values) for values in node_ranges.values()],
    }
)


In [ ]:
runs = []
for sample_count, node_counts in node_ranges.items():
    runs.append(
        scalar_quadrature.run_scalar_monte_carlo(
            FUNCTIONS,
            sample_counts=[sample_count],
            chebyshev_node_counts=node_counts,
            noise_fractions=NOISE_FRACTIONS,
            num_seeds=NUM_SEEDS,
            seed=RANDOM_SEED,
            interval=MIDDLE_INTERVAL,
            evaluation_count=EVALUATION_COUNT,
            lambda1=EXPERIMENT_LAMBDA1,
        )
    )

method_metrics = pd.concat(
    [run.method_metrics for run in runs], ignore_index=True
)
comparisons = pd.concat(
    [run.comparisons for run in runs], ignore_index=True
)
display(pd.DataFrame({"quantity": ["lambda1 Tikhonov D penalty"], "value": [EXPERIMENT_LAMBDA1]}))
comparisons.head()


In [ ]:
for function in FUNCTIONS:
    fig = scalar_quadrature.plot_fixed_sample_comparison(
        comparisons,
        function_name=function.name,
        selected_sample_counts=SAMPLE_COUNTS,
        show_sqrt_sample_count=True,
    )
    display(fig)
    plt.close(fig)


In [ ]:
summary = (
    comparisons.groupby(["function", "sample_count"])[["end_error", "rmse_error", "max_error"]]
    .median()
    .round(6)
)
summary


## Decision-oriented Plotly views

These views use the same comparison dataframe as the heatmaps above. Positive advantage is `trapezoid error - Chebyshev2 error`, so values above zero favor Chebyshev2. The diamond marks the best median RMSE `m`; the light grey dashed line is `sqrt(N)`. The table reports ideal `m` by metric plus a rank-based robust `m`.


In [ ]:
for function in FUNCTIONS:
    display(
        scalar_quadrature.plot_advantage_curves_by_sample_count(
            comparisons,
            function.name,
            selected_sample_counts=SAMPLE_COUNTS,
            metric="rmse_error",
            y_range_min_m=5,
        )
    )
    display(
        scalar_quadrature.plot_robust_m_table(
            comparisons,
            function.name,
            selected_sample_counts=SAMPLE_COUNTS,
        )
    )


In [ ]:
aggregate_summary = (
    comparisons.assign(advantage_rmse=-comparisons["rmse_error"])
    .groupby(["sample_count", "chebyshev_nodes"], as_index=False)
    .agg(
        median_rmse_advantage=("advantage_rmse", "median"),
        win_rate=("advantage_rmse", lambda values: float(np.mean(values > 0.0))),
    )
)
aggregate_summary.sort_values(
    ["sample_count", "median_rmse_advantage"], ascending=[True, False]
).groupby("sample_count").head(3)


## Lambda1 Sweep for N=40

This sweep reruns only the `N=40` experiment over a broad lambda1 grid. Win rate is the fraction of RMSE comparisons where Chebyshev2 beats trapezoid, aggregated across all EuRoC snippets, noise levels, and fitted Chebyshev2 node counts `m`.


In [ ]:
lambda_sweep_runs = []
for lambda1 in LAMBDA1_SWEEP_GRID:
    result = scalar_quadrature.run_scalar_monte_carlo(
        FUNCTIONS,
        sample_counts=[LAMBDA1_SWEEP_SAMPLE_COUNT],
        chebyshev_node_counts=CHEBYSHEV_NODES,
        noise_fractions=NOISE_FRACTIONS,
        num_seeds=NUM_SEEDS,
        seed=RANDOM_SEED,
        interval=MIDDLE_INTERVAL,
        evaluation_count=EVALUATION_COUNT,
        lambda1=float(lambda1),
    )
    lambda_sweep_runs.append(result.comparisons)

lambda_sweep_comparisons = pd.concat(lambda_sweep_runs, ignore_index=True)
lambda_sweep_comparisons = lambda_sweep_comparisons.assign(
    rmse_advantage=-lambda_sweep_comparisons["rmse_error"],
    lambda1_label=lambda_sweep_comparisons["lambda1"].map(lambda value: f"{value:.1e}"),
)

lambda_sweep_summary = (
    lambda_sweep_comparisons.groupby("lambda1", as_index=False)
    .agg(
        win_rate=("rmse_advantage", lambda values: float(np.mean(values > 0.0))),
        median_rmse_advantage=("rmse_advantage", "median"),
        mean_rmse_advantage=("rmse_advantage", "mean"),
        rows=("rmse_advantage", "size"),
    )
    .sort_values(
        ["win_rate", "median_rmse_advantage", "mean_rmse_advantage"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)
BEST_LAMBDA1_FOR_N40 = float(lambda_sweep_summary.loc[0, "lambda1"])

lambda_sweep_summary.head(10)


In [ ]:
lambda_by_m = (
    lambda_sweep_comparisons.groupby(["lambda1", "lambda1_label", "chebyshev_nodes"], as_index=False)
    .agg(
        win_rate=("rmse_advantage", lambda values: float(np.mean(values > 0.0))),
        median_rmse_advantage=("rmse_advantage", "median"),
    )
)
ordered_labels = [f"{value:.1e}" for value in LAMBDA1_SWEEP_GRID]
win_rate_image = (
    lambda_by_m.pivot(index="lambda1_label", columns="chebyshev_nodes", values="win_rate")
    .reindex(ordered_labels)
)

fig = go.Figure(
    data=go.Heatmap(
        z=win_rate_image.to_numpy(dtype=float),
        x=[f"m={int(value)}" for value in win_rate_image.columns],
        y=win_rate_image.index,
        colorscale="Viridis",
        zmin=0.0,
        zmax=1.0,
        colorbar_title="win rate",
    )
)
fig.update_layout(
    title=(
        f"N={LAMBDA1_SWEEP_SAMPLE_COUNT}: lambda1 sweep win rate; "
        f"best lambda1={BEST_LAMBDA1_FOR_N40:.3g}"
    ),
    xaxis_title="Chebyshev2 fitted node count m",
    yaxis_title="lambda1",
    yaxis_autorange="reversed",
    height=520,
    margin=dict(l=90, r=40, t=80, b=60),
)
fig.show()
